In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import joblib
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
# Load the dataset
df = pd.read_excel('ai_job_trends_dataset.xlsx')

print(f" Dataset shape: {df.shape}")

print("\n First 5 rows:")
display(df.head())

print("\n Dataset info:")
df.info()

print("\n Missing values:")
missing = df.isnull().sum()
if missing.sum() == 0:
    print(" No missing values found!")
else:
    print(missing[missing > 0])

print(f"\n Duplicate rows: {df.duplicated().sum()}")

print("\n Statistical summary:")
display(df.describe())



 Dataset shape: (30000, 13)

 First 5 rows:


,Job Title,Industry,Job Status,AI Impact Level,Median Salary (USD),Required Education,Experience Required (Years),Job Openings (2024),Projected Openings (2030),Remote Work Ratio (%),Automation Risk (%),Location,Gender Diversity (%)
0,Investment analyst,IT,Increasing,Moderate,42109.76,Masterâ€™s Degree,5,1515,6342,55.96,28.28,UK,44.63
1,"Journalist, newspaper",Manufacturing,Increasing,Moderate,132298.57,Masterâ€™s Degree,15,1243,6205,16.81,89.71,USA,66.39
2,Financial planner,Finance,Increasing,Low,143279.19,Bachelorâ€™s Degree,4,3338,1154,91.82,72.97,Canada,41.13
3,Legal secretary,Healthcare,Increasing,High,97576.13,Associate Degree,15,7173,4060,1.89,99.94,Australia,65.76
4,Aeronautical engineer,IT,Increasing,Low,60956.63,Masterâ€™s Degree,13,5944,7396,53.76,37.65,Germany,72.57



 Dataset info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 13 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Job Title                    30000 non-null  object 
 1   Industry                     30000 non-null  object 
 2   Job Status                   30000 non-null  object 
 3   AI Impact Level              30000 non-null  object 
 4   Median Salary (USD)          30000 non-null  float64
 5   Required Education           30000 non-null  object 
 6   Experience Required (Years)  30000 non-null  int64  
 7   Job Openings (2024)          30000 non-null  int64  
 8   Projected Openings (2030)    30000 non-null  int64  
 9   Remote Work Ratio (%)        30000 non-null  float64
 10  Automation Risk (%)          30000 non-null  float64
 11  Location                     30000 non-null  object 
 12  Gender Diversity (%)         30000 non-null  float64
dtype

,Median Salary (USD),Experience Required (Years),Job Openings (2024),Projected Openings (2030),Remote Work Ratio (%),Automation Risk (%),Gender Diversity (%)
count,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000
mean,90119.965639,10.051433,5039.640833,5074.218100,49.836431,50.154229,49.979660
std,34412.013953,6.060678,2861.009654,2866.550722,28.966688,28.754889,17.274665
min,30001.860000,0.000000,100.000000,100.000000,0.000000,0.000000,20.000000
25%,60500.702500,5.000000,2570.000000,2586.750000,24.570000,25.400000,35.070000
50%,90274.115000,10.000000,5034.000000,5106.500000,49.570000,50.020000,49.885000
75%,119454.710000,15.000000,7527.000000,7573.000000,75.100000,75.030000,64.910000
max,149998.500000,20.000000,10000.000000,10000.000000,100.000000,99.990000,80.000000


In [4]:
education_mapping = {
    "Masterâ€™s Degree": "Master's Degree",
    "Bachelorâ€™s Degree": "Bachelor's Degree",
    "Associate Degree": "Associate Degree",
    "High School": "High School",
    "PhD": "PhD"
}

if 'Required Education' in df.columns:
    df['Required Education'] = df['Required Education'].replace(education_mapping)
    print(" Fixed encoding issues in 'Required Education' column")
    print(f" Unique values now: {df['Required Education'].unique()}")
else:
    print("   'Required Education' column not found!")



 Fixed encoding issues in 'Required Education' column
 Unique values now: ["Master's Degree" "Bachelor's Degree" 'Associate Degree' 'High School'
 'PhD']


In [5]:
y = df['AI Impact Level'].copy()

print(f" Target variable: AI Impact Level")
print(f" Classes: {y.unique()}")

print("\n Class distribution:")
display(y.value_counts().to_frame(name="Count"))


 Target variable: AI Impact Level
 Classes: ['Moderate' 'Low' 'High']

 Class distribution:


,Count
AI Impact Level,
Moderate,10042
High,10005
Low,9953


In [6]:
categorical_cols = ['Job Title', 'Industry', 'Job Status', 'Required Education', 'Location']

numerical_cols = [
    'Median Salary (USD)', 'Experience Required (Years)', 'Job Openings (2024)',
    'Projected Openings (2030)', 'Remote Work Ratio (%)', 
    'Automation Risk (%)', 'Gender Diversity (%)'
]

print(f" Categorical features ({len(categorical_cols)}):")
for col in categorical_cols:
    print(f"    {col} → {df[col].nunique()} unique values")

print(f"\n Numerical features ({len(numerical_cols)}):")
for col in numerical_cols:
    print(f"    {col} → [{df[col].min():.2f} to {df[col].max():.2f}]")


 Categorical features (5):
    Job Title → 639 unique values
    Industry → 8 unique values
    Job Status → 2 unique values
    Required Education → 5 unique values
    Location → 8 unique values

 Numerical features (7):
    Median Salary (USD) → [30001.86 to 149998.50]
    Experience Required (Years) → [0.00 to 20.00]
    Job Openings (2024) → [100.00 to 10000.00]
    Projected Openings (2030) → [100.00 to 10000.00]
    Remote Work Ratio (%) → [0.00 to 100.00]
    Automation Risk (%) → [0.00 to 99.99]
    Gender Diversity (%) → [20.00 to 80.00]


In [7]:
def encode_categorical(df, categorical_cols):
    df_encoded = df.copy()
    label_encoders = {}
    
    for col in categorical_cols:
        le = LabelEncoder()
        df_encoded[col] = le.fit_transform(df_encoded[col])
        label_encoders[col] = le
        print(f" Encoded '{col}' → {len(le.classes_)} classes")
    
    return df_encoded, label_encoders

# Apply encoding
df_encoded, label_encoders = encode_categorical(df, categorical_cols)

print(f"\n Total columns encoded: {len(label_encoders)}")


 Encoded 'Job Title' → 639 classes
 Encoded 'Industry' → 8 classes
 Encoded 'Job Status' → 2 classes
 Encoded 'Required Education' → 5 classes
 Encoded 'Location' → 8 classes

 Total columns encoded: 5


In [8]:
drop_cols = ['AI Impact Level', 'Automation Risk (%)']
X = df_encoded.drop(columns=drop_cols)

print(f" Feature matrix shape: {X.shape}")
print(f" Target shape: {y.shape}")

print("\n Final features used for modeling:")
for i, col in enumerate(X.columns, 1):
    print(f"   {i}. {col}")


 Feature matrix shape: (30000, 11)
 Target shape: (30000,)

 Final features used for modeling:
   1. Job Title
   2. Industry
   3. Job Status
   4. Median Salary (USD)
   5. Required Education
   6. Experience Required (Years)
   7. Job Openings (2024)
   8. Projected Openings (2030)
   9. Remote Work Ratio (%)
   10. Location
   11. Gender Diversity (%)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

print(f" Training samples : {X_train.shape[0]}")
print(f" Testing samples  : {X_test.shape[0]}")
print(f" Number of features: {X_train.shape[1]}")

print("\n Training class distribution:")
display(y_train.value_counts().to_frame(name="Count"))

print("\n Test class distribution:")
display(y_test.value_counts().to_frame(name="Count"))



 Training samples : 24000
 Testing samples  : 6000
 Number of features: 11

 Training class distribution:


,Count
AI Impact Level,
Moderate,8034
High,8004
Low,7962



 Test class distribution:


,Count
AI Impact Level,
Moderate,2008
High,2001
Low,1991


In [10]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

print(f" Scaled training data shape: {X_train_scaled.shape}")
print(f" Scaled test data shape: {X_test_scaled.shape}")
print(" StandardScaler applied successfully")


 Scaled training data shape: (24000, 11)
 Scaled test data shape: (6000, 11)
 StandardScaler applied successfully


In [11]:
def save_preprocessed_data(output_dir='../data'):
    os.makedirs(output_dir, exist_ok=True)
    
    X_train_scaled.to_csv(f'{output_dir}/X_train.csv', index=False)
    X_test_scaled.to_csv(f'{output_dir}/X_test.csv', index=False)
    y_train.to_csv(f'{output_dir}/y_train.csv', index=False, header=['AI Impact Level'])
    y_test.to_csv(f'{output_dir}/y_test.csv', index=False, header=['AI Impact Level'])
    df.to_csv(f'{output_dir}/cleaned_full_dataset.csv', index=False)
    
    joblib.dump(scaler, f'{output_dir}/scaler.pkl')
    joblib.dump(label_encoders, f'{output_dir}/label_encoders.pkl')
    
    print(f" All files saved to folder: {output_dir}/")
    print("    X_train.csv, X_test.csv, y_train.csv, y_test.csv")
    print("    cleaned_full_dataset.csv")
    print("    scaler.pkl, label_encoders.pkl")

# Save everything
save_preprocessed_data()


 All files saved to folder: ../data/
    X_train.csv, X_test.csv, y_train.csv, y_test.csv
    cleaned_full_dataset.csv
    scaler.pkl, label_encoders.pkl


In [12]:
print(f"\n Dataset Overview:")
print(f"   Total samples     : {len(df)}")
print(f"   Total features    : {df.shape[1]}")
print(f"   Target classes    : {y.nunique()}")

print(f"\n Train / Test Split:")
print(f"   Training samples  : {len(X_train)}")
print(f"   Testing samples   : {len(X_test)}")
print(f"   Number of features: {X_train.shape[1]}")


 Dataset Overview:
   Total samples     : 30000
   Total features    : 13
   Target classes    : 3

 Train / Test Split:
   Training samples  : 24000
   Testing samples   : 6000
   Number of features: 11
